# 01 — Từ CSV đến Bronze

### 1. Xác định đường dẫn nguồn

Repo cung cấp `data/delivery_yt.csv`. Notebook và Prefect ưu tiên đối tượng
`s3a://lakehouse/delivery_yt.csv`. Khi thiếu đối tượng đó, hàm dùng file
`file:///home/jovyan/data/delivery_yt.csv` trong Jupyter. Hai biến
`SOURCE_PATH` và `LOCAL_SOURCE` cấu hình hai đường dẫn.

Chạy trong Jupyter Docker với MinIO sẵn sàng.

In [ ]:
import sys

sys.path.insert(0, "/home/jovyan/pipeline")
import steps

spark = steps.get_spark("Delivery - Bronze")

### 2. Đọc 17 cột, kiểm tra khóa và ghi Delta

`steps.ingest_bronze()` đọc các cột dưới dạng chuỗi, kiểm tra `order_id` rồi
ghi `delivery_bronze`. Hàm đối soát khóa vừa ghi với CSV. Trường `source`
ở kết quả cho biết đường dẫn thực tế đã đọc.

In [1]:
result = steps.ingest_bronze(spark)
result

{'source': 's3a://lakehouse/delivery_yt.csv', 'bronze_rows': 206431}

### 3. Xem dữ liệu đã ghi

Các cột thời gian ở Bronze giữ định dạng của CSV. Notebook 02 sẽ đổi kiểu.

In [ ]:
bronze = spark.read.format("delta").load(steps.BRONZE)
print(steps.BRONZE)
bronze.select("order_id", "courier_id", "aoi_id", "accept_time", "delivery_time").show(5, truncate=False)